In [1]:
import pandas as pd
import numpy as np
import pyarrow.dataset as ds

### build the cleaned csv

In [2]:
DATA_DIR = 'data'

job_cols = ['JOB_HASH', 'CITY', 'COMPANY_NAME', 'TITLE', 'URL', 'ZIP']
salary_cols = ['JOB_HASH', 'DATE_TIME', 'EXTRACTED_VALUE_HIGH', 'EXTRACTED_VALUE_LOW', 'FREQUENCY',
               'NORMALIZED_ANNUAL_HIGH_USD', 'NORMALIZED_ANNUAL_LOW_USD', 'NORMALIZED_FREQUENCY']

jobs = pd.read_csv(f'{DATA_DIR}/job-records_combined.csv', usecols=job_cols)
job_hashes = set(jobs['JOB_HASH'])

salary = pd.read_csv(f'{DATA_DIR}/extracted-salary_combined.csv', usecols=salary_cols)

jobs.shape, salary.shape

((1281471, 6), (23720459, 8))

### keeping the job hash for the parquet file

In [3]:
df = jobs.merge(salary, on='JOB_HASH', how='inner')
df = df[job_cols + salary_cols[1:]]
df.shape

(375056, 13)

In [4]:
df.head()

,JOB_HASH,CITY,COMPANY_NAME,TITLE,URL,ZIP,DATE_TIME,EXTRACTED_VALUE_HIGH,EXTRACTED_VALUE_LOW,FREQUENCY,NORMALIZED_ANNUAL_HIGH_USD,NORMALIZED_ANNUAL_LOW_USD,NORMALIZED_FREQUENCY
0,104fb04ee3865bddd564238fcd3179b1,collierville,FedEx Corp,Electronics Technician I,https://careers.fedex.com/electronics-technici...,38017,2025-03-05,32.1,16.92,hourly,66768.0,35193.6,hourly
1,105022dec4ebdc2bce7aeeaf14efcc3b,knoxville,Walmart Inc,(USA) Personal Shopper - Sam's,https://careers.walmart.com/us/jobs/WD2118902,37902,2025-03-06,24.0,17.00,hourly,49920.0,35360.0,hourly
2,1050387b84b4b393ef193f4ee4d933b3,signal mountain,AccentCare Inc,Private Caregiver,https://careers.accentcare.com/jobs/39256?lang...,37377,2024-03-10,NaN,15.00,hourly,NaN,31200.0,hourly
3,105070d0d9c684055f1f0b036eda5626,NaN,CVS Health Corp,Lead Director Finance Analytics,https://cvshealth.wd1.myworkdayjobs.com/CVS_He...,37201,2026-07-13,231540.0,100000.00,yearly,231540.0,100000.0,yearly
4,10509be4341ffa3fffe0975a2df60e8a,memphis,DHL Group,Order Filler Picker,https://careers.dhl.com/amer/en/job/10281586,38103,2024-10-18,NaN,18.55,hourly,NaN,38584.0,hourly


### add the ONET codes

In [5]:
onet = ds.dataset(f'{DATA_DIR}/onet-taxonomy_combined.snappy.parquet').to_table(
    filter=ds.field('JOB_HASH').isin(df['JOB_HASH'].unique())
).to_pandas()

df = df.merge(onet, on='JOB_HASH', how='left')
df.head()

,JOB_HASH,CITY,COMPANY_NAME,TITLE,URL,ZIP,DATE_TIME,EXTRACTED_VALUE_HIGH,EXTRACTED_VALUE_LOW,FREQUENCY,NORMALIZED_ANNUAL_HIGH_USD,NORMALIZED_ANNUAL_LOW_USD,NORMALIZED_FREQUENCY,ONET_OCCUPATION_CODE
0,104fb04ee3865bddd564238fcd3179b1,collierville,FedEx Corp,Electronics Technician I,https://careers.fedex.com/electronics-technici...,38017,2025-03-05,32.1,16.92,hourly,66768.0,35193.6,hourly,17-3023.00
1,105022dec4ebdc2bce7aeeaf14efcc3b,knoxville,Walmart Inc,(USA) Personal Shopper - Sam's,https://careers.walmart.com/us/jobs/WD2118902,37902,2025-03-06,24.0,17.00,hourly,49920.0,35360.0,hourly,31-1122.00
2,1050387b84b4b393ef193f4ee4d933b3,signal mountain,AccentCare Inc,Private Caregiver,https://careers.accentcare.com/jobs/39256?lang...,37377,2024-03-10,NaN,15.00,hourly,NaN,31200.0,hourly,31-1122.00
3,105070d0d9c684055f1f0b036eda5626,NaN,CVS Health Corp,Lead Director Finance Analytics,https://cvshealth.wd1.myworkdayjobs.com/CVS_He...,37201,2026-07-13,231540.0,100000.00,yearly,231540.0,100000.0,yearly,11-3031.00
4,10509be4341ffa3fffe0975a2df60e8a,memphis,DHL Group,Order Filler Picker,https://careers.dhl.com/amer/en/job/10281586,38103,2024-10-18,NaN,18.55,hourly,NaN,38584.0,hourly,53-7065.00


In [6]:
# onet groups
major_groups = {
    '11': 'Management',
    '13': 'Business and Financial Operations',
    '15': 'Computer and Mathematical',
    '17': 'Architecture and Engineering',
    '19': 'Life, Physical, and Social Science',
    '21': 'Community and Social Service',
    '23': 'Legal',
    '25': 'Educational Instruction and Library',
    '27': 'Arts, Design, Entertainment, Sports, and Media',
    '29': 'Healthcare Practitioners and Technical',
    '31': 'Healthcare Support',
    '33': 'Protective Service',
    '35': 'Food Preparation and Serving Related',
    '37': 'Building and Grounds Cleaning and Maintenance',
    '39': 'Personal Care and Service',
    '41': 'Sales and Related',
    '43': 'Office and Administrative Support',
    '45': 'Farming, Fishing, and Forestry',
    '47': 'Construction and Extraction',
    '49': 'Installation, Maintenance, and Repair',
    '51': 'Production',
    '53': 'Transportation and Material Moving',
    '55': 'Military Specific',
}

df['MAJOR_GROUP'] = df['ONET_OCCUPATION_CODE'].str[:2].map(major_groups)
df['MAJOR_GROUP'].value_counts()

MAJOR_GROUP
Sales and Related                                 54388
Transportation and Material Moving                40611
Management                                        36845
Healthcare Practitioners and Technical            35186
Office and Administrative Support                 31586
Business and Financial Operations                 28990
Food Preparation and Serving Related              18913
Installation, Maintenance, and Repair             17124
Computer and Mathematical                         17047
Protective Service                                13120
Healthcare Support                                12799
Arts, Design, Entertainment, Sports, and Media    12436
Production                                        11548
Building and Grounds Cleaning and Maintenance      9161
Community and Social Service                       7023
Educational Instruction and Library                5423
Architecture and Engineering                       5251
Personal Care and Service           

In [7]:
# now map the code to the proper occupation
occupations = pd.read_csv('All_Occupations.csv')
code_to_occupation = occupations.set_index('Code')['Occupation']

df['ONET_OCCUPATION_CODE'] = df['ONET_OCCUPATION_CODE'].map(code_to_occupation)
df = df.rename(columns={'ONET_OCCUPATION_CODE': 'OCCUPATION'})
df['OCCUPATION'].value_counts().head(10)

OCCUPATION
First-Line Supervisors of Retail Sales Workers    15784
Retail Salespersons                               13991
Customer Service Representatives                  12587
Heavy and Tractor-Trailer Truck Drivers           11045
Cashiers                                          10034
Security Guards                                    9620
Merchandise Displayers and Window Trimmers         8834
Registered Nurses                                  7407
General and Operations Managers                    6982
Pharmacy Technicians                               6934
Name: count, dtype: int64

In [8]:
# drop any rows that don't have both major group and occupation
df = df.dropna(subset=['MAJOR_GROUP', 'OCCUPATION'])
df.shape

(374505, 15)

In [9]:
# save it
df.to_csv(f'{DATA_DIR}/JobDataClean.csv', index=False)

### classify major group from job title

In [10]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

INCLUDE_COMPANY = False

titles = df[['TITLE', 'COMPANY_NAME', 'OCCUPATION', 'MAJOR_GROUP']].copy()
titles['TITLE_KEY'] = titles['TITLE'].str.lower().str.strip()
titles['TEXT'] = titles['TITLE'] + ' | ' + titles['COMPANY_NAME'] if INCLUDE_COMPANY else titles['TITLE']

# split by title so test titles are never seen in training
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(titles, groups=titles['TITLE_KEY']))

train_rows, test = titles.iloc[train_idx], titles.iloc[test_idx]

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)

fit_idx, val_idx = next(gss_val.split(train_rows, groups=train_rows['TITLE_KEY']))

fit_rows, val = train_rows.iloc[fit_idx], train_rows.iloc[val_idx]

# train on one row per unique text so repeated postings don't dominate
train = (fit_rows.groupby(fit_rows['TEXT'].str.lower().str.strip())
         .agg(TEXT=('TEXT', 'first'),
              OCCUPATION=('OCCUPATION', lambda s: s.mode()[0]),
              MAJOR_GROUP=('MAJOR_GROUP', lambda s: s.mode()[0]))
         .reset_index(drop=True))

# unique titles are better, mainly because there's lot of duplicates
# like "shift lead" and such
test_unique = test.drop_duplicates('TITLE_KEY')
occupation_to_group = df.groupby('OCCUPATION')['MAJOR_GROUP'].first()


def report(name, y_true, y_pred):
    print(f"{name}: accuracy={accuracy_score(y_true, y_pred):.3f}  macro F1={f1_score(y_true, y_pred, average='macro'):.3f}")


len(train), len(val), len(test), len(test_unique)

(74592, 27559, 78937, 20721)

### TF-IDF + linear SVM

In [11]:
baseline = make_pipeline(
    make_union(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2),
               TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), sublinear_tf=True, min_df=3)),
    LinearSVC(C=0.5),
)

baseline.fit(train['TEXT'].str.lower(), train['MAJOR_GROUP'])

report('test rows', test['MAJOR_GROUP'], baseline.predict(test['TEXT'].str.lower()))

report('unique test titles', test_unique['MAJOR_GROUP'], baseline.predict(test_unique['TEXT'].str.lower()))

test rows: accuracy=0.903  macro F1=0.832
unique test titles: accuracy=0.865  macro F1=0.815


In [12]:
title_to_predict = "Farmer"

In [13]:
def predict_title(title: str, company: str = None):
    text = f"{title} | {company}" if INCLUDE_COMPANY and company is not None else title
    return baseline.predict([text.lower()])[0]


predicted_title = predict_title(title_to_predict)

In [14]:
print(f"{title_to_predict} is part of the '{predicted_title}' group.")

Farmer is part of the 'Farming, Fishing, and Forestry' group.


### where the jobs are

In [15]:
# zip code lookup from geonames, gives place, county, and lat/lon for every US zip
geo_cols = ['country', 'zip', 'place', 'state', 'state_code', 'county', 'county_code',
            'admin3', 'admin3_code', 'lat', 'lon', 'accuracy']
geo = pd.read_csv(f'{DATA_DIR}/geo/US.txt', sep='\t', header=None, names=geo_cols, dtype={'zip': str})
zip_info = geo.drop_duplicates('zip').set_index('zip')

# some zips got saved as floats like 37056.0 and some are zip+4, so just keep the first 5 digits
# i'm not writing regex thank you stack overflow
df['ZIP5'] = df['ZIP'].astype(str).str.extract(r'^(\d{5})')[0]

# get rid of junk and out of state zips like 00000
df.loc[df['ZIP5'].map(zip_info['state_code']) != 'TN', 'ZIP5'] = np.nan

# fill missing zips with the most common zip for that city
city_zip = df.dropna(subset=['ZIP5', 'CITY']).groupby('CITY')['ZIP5'].agg(lambda s: s.mode()[0])
df['ZIP5'] = df['ZIP5'].fillna(df['CITY'].map(city_zip))

df['PLACE'] = df['ZIP5'].map(zip_info['place'])
df['COUNTY'] = df['ZIP5'].map(zip_info['county'])

# postings that just say "Nashville, TN" all get the downtown zip, so one zip has most of each big city postings
# like 37201, 38103, 37902, 37402 mark those as city-wide instead of a neighborhood
place_counts = df.groupby(['PLACE', 'ZIP5']).size()
place_share = place_counts / place_counts.groupby(level='PLACE').transform('sum')
zips_in_place = place_counts.groupby(level='PLACE').transform('size')

city_default_zips = set(place_share[(place_share > 0.5) & (zips_in_place >= 10)].index.get_level_values('ZIP5'))

zip_totals = df.groupby('ZIP5').size()

print(f"rows with a TN zip: {df['ZIP5'].notna().mean():.2%}, city-wide zips: {sorted(city_default_zips)}")

rows with a TN zip: 98.84%, city-wide zips: ['37201', '37402', '37902', '38103']


In [16]:
def top_places(jobs, weight=None, n=10, min_zip_jobs=50):
    # every job counts as 1 unless it's weighted (like by similarity)
    w = pd.Series(1.0 if weight is None else weight, index=jobs.index, dtype=float)
    counts = w.groupby(jobs['ZIP5']).sum()

    places = pd.DataFrame({'jobs': counts.round(1), 'zip_total_jobs': zip_totals.reindex(counts.index)})

    # skip tiny zips, a zip with 3 postings can look super concentrated
    places = places[places['zip_total_jobs'] >= min_zip_jobs]

    # concentration > 1 means jobs are a bigger share of that zip's postings than of TN overall
    places['concentration'] = ((places['jobs'] / places['zip_total_jobs']) / (w.sum() / len(df))).round(2)

    places['place'] = zip_info['place'].reindex(places.index)
    places.loc[places.index.isin(city_default_zips), 'place'] += ' (city-wide)'
    places['county'] = zip_info['county'].reindex(places.index)

    return places.sort_values('jobs', ascending=False).head(n)[['place', 'county', 'jobs', 'zip_total_jobs', 'concentration']]


def top_places_for_group(group, **kwargs):
    return top_places(df[df['MAJOR_GROUP'] == group], **kwargs)


print(f"Where '{predicted_title}' jobs are:")
top_places_for_group(predicted_title)

Where 'Farming, Fishing, and Forestry' jobs are:


,place,county,jobs,zip_total_jobs,concentration
ZIP5,,,,,
37726,Deer Lodge,Morgan,266.0,367,360.96
37083,Lafayette,Macon,108.0,356,151.08
38059,Newbern,Dyer,53.0,520,50.76
37086,La Vergne,Rutherford,40.0,3220,6.19
38103,Memphis (city-wide),Shelby,34.0,39524,0.43
37886,Walland,Blount,24.0,1016,11.76
38343,Humboldt,Gibson,22.0,608,18.02
37201,Nashville (city-wide),Davidson,21.0,67617,0.15
38261,Union City,Obion,15.0,390,19.15


#### places with related jobs, via title embeddings

In [17]:
import torch
from sentence_transformers import SentenceTransformer

# small embedding model
embedder = SentenceTransformer('BAAI/bge-small-en-v1.5', device='cuda' if torch.cuda.is_available() else 'cpu')

unique_titles = df['TITLE'].str.strip().drop_duplicates().reset_index(drop=True)
title_embeddings = embedder.encode(unique_titles.tolist(), batch_size=1024, normalize_embeddings=True, convert_to_tensor=True, show_progress_bar=True)


def similar_titles(query, min_similarity=0.8, max_titles=2000):
    q = embedder.encode([query], normalize_embeddings=True, convert_to_tensor=True)

    # embeddings are normalized, so the dot product is the cosine similarity
    sims = (title_embeddings @ q.T).squeeze(1)
    top = torch.topk(sims, k=min(max_titles, len(sims)))

    out = pd.Series(top.values.float().cpu().numpy(), index=unique_titles.iloc[top.indices.cpu().numpy()].values)

    # always keep anything within 0.05 of the best match
    cutoff = min(min_similarity, out.iloc[0] - 0.05)
    return out[out >= cutoff]


def top_places_for_similar_jobs(query, min_similarity=0.8, **kwargs):
    sims = similar_titles(query, min_similarity)

    # grab every posting with one of the similar titles, weighted by how similar it is
    stripped = df['TITLE'].str.strip()
    jobs = df[stripped.isin(sims.index)]

    return top_places(jobs, weight=stripped[jobs.index].map(sims).values, **kwargs)

/home/cayden/miniconda3/envs/DataDive/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 103/103 [00:06<00:00, 15.10it/s]


In [18]:
matches = similar_titles(title_to_predict)

print(f"{len(matches)} titles similar to '{title_to_predict}':")
print(matches.head(15).round(3).to_string())

print(f"\nWhere jobs like '{title_to_predict}' are:")
top_places_for_similar_jobs(title_to_predict)

6 titles similar to 'Farmer':
Gardener                       0.789
Farm Supervisor                0.765
Tractor Operator               0.759
Agriculture Teacher            0.756
(Seasonal) Tractor Operator    0.753
Barn Worker                    0.750

Where jobs like 'Farmer' are:


,place,county,jobs,zip_total_jobs,concentration
ZIP5,,,,,
37058,Dover,Stewart,3.1,148,536.63
37601,Johnson City,Washington,2.4,8445,7.28
38462,Hohenwald,Lewis,1.5,157,244.78
37064,Franklin,Williamson,1.5,7730,4.97
37130,Murfreesboro,Rutherford,0.8,8614,2.38
37129,Murfreesboro,Rutherford,0.8,2184,9.38
37660,Kingsport,Sullivan,0.8,1291,15.88
38237,Martin,Weakley,0.8,739,27.73
